# 00 · Cruce de nombres entre fuentes

Este cuaderno decide **qué jugador es quién** en las tres fuentes del proyecto:

- mi fichero de posiciones (`posiciones_laliga_2526.csv`)
- Understat (`jugadores_laliga_2526.csv`): minutos, partidos y métricas ofensivas
- WhoScored/Opta (alineaciones y eventos): identificador de jugador y acciones

Todas las decisiones de cruce están escritas a mano en las listas de este cuaderno, para
que se puedan auditar. El resultado se guarda en ficheros versionados que lee el cuaderno 01:

| Fichero | Contenido |
|---|---|
| `data/mapa_nombres.csv` | un jugador por fila, con su nombre en cada fuente y su identificador de WhoScored |
| `data/eventos_por_jugador.csv` | acciones de Opta contadas por identificador de WhoScored |
| `data/posiciones_alineaciones.csv` | posición deducida de las alineaciones para quien no está en mi fichero |
| `data/sin_minutos.csv` | jugadores de mi fichero de posiciones que no jugaron en LaLiga |
| `data/cruces_descartados.csv` | cruces dudosos que no se aplican y su motivo |

El único fichero que no se versiona es `data/bruto/eventos_laliga_2526.parquet` (16 MB).

In [1]:
# Instalamos las librerías del cuaderno; si ya están instaladas no cambia nada
%pip install -q pandas pyarrow unidecode

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos pandas para trabajar con tablas
import pandas as pd
# Importamos unidecode para comparar nombres sin acentos
from unidecode import unidecode

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Ensanchamos la salida para que las tablas no se partan
pd.set_option("display.width", 250)
# Mostramos todas las filas de las tablas de control
pd.set_option("display.max_rows", 300)

## 1. Normalización de nombres y equipos

In [3]:
# Función que deja un nombre en minúsculas, sin acentos, puntos ni guiones
def normalizar(nombre):
    # Quitamos acentos, pasamos a minúsculas y eliminamos puntos y guiones
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    # Compactamos los espacios repetidos
    return " ".join(n.split())


# Nombres de equipo de WhoScored y de la altura defensiva que no coinciden con mi fichero
EQUIPOS = {
    # WhoScored escribe "Atletico" a secas
    "Atletico": "Atletico Madrid",
    # WhoScored escribe el nombre oficial completo del Alavés
    "Deportivo Alaves": "Alaves",
}

## 2. Carga de las fuentes

In [4]:
# Fichero propio con la posición de cada jugador
posiciones = pd.read_csv("data/origen/posiciones_laliga_2526.csv")
# Jugadores de Understat: minutos, partidos y métricas ofensivas
understat = pd.read_csv("data/origen/jugadores_laliga_2526.csv")
# Tabla de métricas del TFM: guarda el cruce posiciones-Understat ya validado
metricas_tfm = pd.read_csv("data/origen/metricas_jugadores_laliga_2526.csv")
# Alineaciones de WhoScored: un jugador convocado por partido, con su identificador
alineaciones = pd.read_csv("data/origen/alineaciones_laliga_2526.csv")

# Columnas de eventos que necesitamos para contar acciones
COLUMNAS_EVENTOS = ["game_id", "team", "player_id", "type", "outcome_type", "qualifiers", "is_touch", "x", "y"]
# Eventos de Opta (fichero en bruto, no versionado)
eventos = pd.read_parquet("data/bruto/eventos_laliga_2526.parquet", columns=COLUMNAS_EVENTOS)

# Unificamos el nombre de equipo de las alineaciones
alineaciones["equipo"] = alineaciones["team"].replace(EQUIPOS)
# Unificamos el nombre de equipo de los eventos
eventos["equipo"] = eventos["team"].replace(EQUIPOS)

# Equipos tal y como aparecen en mi fichero de posiciones
EQUIPOS_LIGA = set(posiciones["equipo"])
# Comprobamos que las cuatro fuentes usan ya los mismos 20 equipos
for nombre, tabla, col in [("understat", understat, "team"), ("alineaciones", alineaciones, "equipo"),
                           ("eventos", eventos, "equipo")]:
    # Si algún equipo no cuadra, paramos aquí
    assert set(tabla[col]) == EQUIPOS_LIGA, f"equipos distintos en {nombre}"
# Mostramos el resultado de la comprobación
print("equipos de la liga:", len(EQUIPOS_LIGA), "· mismos nombres en las cuatro fuentes")

equipos de la liga: 20 · mismos nombres en las cuatro fuentes


## 3. Pares posiciones ↔ Understat

**Directos (481).** Son los que ya cruzó el TFM. Se toman de la tabla de métricas del TFM, que guarda
el nombre en las dos fuentes (`jugador` y `player`).

In [5]:
# Nos quedamos con el par de nombres y el equipo de cada jugador del TFM
pares = metricas_tfm[["jugador", "equipo", "player"]].rename(
    # Renombramos a los nombres de columna del mapa
    columns={"jugador": "nombre_posiciones", "player": "nombre_understat"})
# En los directos el equipo de Understat es el mismo que el de posiciones
pares["equipo_understat"] = pares["equipo"]
# Marcamos su origen
pares["origen"] = "directo"
# Mostramos cuántos son
print("pares directos:", len(pares))

pares directos: 481


**Recuperados por nombre (36).** Mismo equipo, nombre distinto, y los partidos con eventos del nombre
de posiciones coinciden con los partidos de Understat. Confirmados por David.

In [6]:
# (nombre en posiciones, equipo, nombre en Understat)
RECUPERADOS_36 = [
    ("Abde Rebbach", "Alaves", "Abderrahmane Rebbach"),
    ("Mariano Díaz", "Alaves", "Mariano"),
    ("Toni Martínez", "Alaves", "Antonio Martínez"),
    ("Youssef Enríquez", "Alaves", "Youssef Lekhedim"),
    ("Nico Serrano", "Athletic Club", "Nicolás Serrano"),
    ("Robert Navarro", "Athletic Club", "Roberto Navarro"),
    ("Yuri Berchiche", "Athletic Club", "Yuri"),
    ("José María Giménez", "Atletico Madrid", "Giménez"),
    ("Nico González", "Atletico Madrid", "Nicolás González"),
    ("Alejandro Balde", "Barcelona", "Álex Balde"),
    ("Fer López", "Celta Vigo", "Fernando López"),
    ("Javi Rodríguez", "Celta Vigo", "Javier Rodríguez"),
    ("Javi Rueda", "Celta Vigo", "Javier Rueda"),
    ("Manu Fernández", "Celta Vigo", "Manuel Fernández"),
    ("Abdel Abqar", "Getafe", "Abdelkabir Abqar"),
    ("Allan Nyom", "Getafe", "Nyom"),
    ("Azzedine Ounahi", "Girona", "Azz-Eddine Ounahi"),
    ("José Luis Morales", "Levante", "Morales"),
    ("Manu Sánchez", "Levante", "Manuel Sánchez"),
    ("Pacha Espino", "Rayo Vallecano", "Alfonso Espino"),
    ("Abde Ezzalzouli", "Real Betis", "Abdessamad Ezzalzouli"),
    ("Chimy Ávila", "Real Betis", "Ezequiel Ávila"),
    ("Cucho Hernández", "Real Betis", "Juan Camilo Hernández"),
    ("Dani Carvajal", "Real Madrid", "Daniel Carvajal"),
    ("Raúl Asencio", "Real Madrid", "Raúl"),
    ("Álvaro Carreras", "Real Madrid", "Álvaro Fernández"),
    ("Javi López", "Real Oviedo", "Javier López"),
    ("Santi Cazorla", "Real Oviedo", "Santiago Cazorla"),
    ("Tanguy Nianzou", "Sevilla", "Nianzou Kouassi"),
    ("Peque Fernández", "Sevilla", "Peque"),
    ("Dani Raba", "Valencia", "Raba"),
    ("Javi Guerra", "Valencia", "Javier Guerra"),
    ("Alex Freeman", "Villarreal", "Alexander Freeman"),
    ("Alfonso Pedraza", "Villarreal", "Alfonso"),
    ("Dani Parejo", "Villarreal", "Daniel Parejo"),
    ("Santi Comesaña", "Villarreal", "Santiago Comesaña"),
]

# Pasamos la lista a tabla
rec36 = pd.DataFrame(RECUPERADOS_36, columns=["nombre_posiciones", "equipo", "nombre_understat"])
# El equipo de Understat es el mismo
rec36["equipo_understat"] = rec36["equipo"]
# Marcamos su origen
rec36["origen"] = "recuperado_36"

**Traspasados (6).** Understat pone toda la temporada bajo un solo club. El equipo del modelo es el de
mi fichero de posiciones (club actual); minutos y partidos, los de Understat.

In [7]:
# (nombre en posiciones, equipo en posiciones, nombre en Understat, equipo en Understat)
TRASPASADOS = [
    ("Fran Beltrán", "Girona", "Fran Beltrán", "Celta Vigo"),
    ("Javi Galán", "Osasuna", "Javi Galán", "Atletico Madrid"),
    ("Umar Sadiq", "Valencia", "Umar Sadiq", "Real Sociedad"),
    ("Yangel Herrera", "Real Sociedad", "Yangel Herrera", "Girona"),
    ("Alfon González", "Villarreal", "Alfon", "Sevilla"),
    ("Sergio Lozano", "Rayo Vallecano", "Sergio Lozano", "Levante"),
]
# Pasamos la lista a tabla
trasp = pd.DataFrame(TRASPASADOS, columns=["nombre_posiciones", "equipo", "nombre_understat", "equipo_understat"])
# Marcamos su origen
trasp["origen"] = "traspasado"

# Juntamos los tres grupos de pares
pares = pd.concat([pares, rec36, trasp], ignore_index=True)

**Cruces de posiciones descartados (2).** No se aplican: el jugador de posiciones queda fuera y su fila
de Understat entra por el grupo "sin posición", con los eventos vacíos.

In [8]:
# (nombre en posiciones, equipo, nombre propuesto en Understat, motivo)
DESCARTADOS_POSICIONES = [
    ("Carlos Protesoni", "Alaves", "Carlos Benavídez",
     "solo comparten el nombre de pila; 13 partidos en Understat y 13 con eventos"),
    ("Yago Santiago", "Elche", "Yago Alonso",
     "solo comparten el nombre de pila; 9 partidos en Understat y 10 con eventos"),
]

## 4. Comprobación: cada jugador aparece una sola vez

In [9]:
# Añadimos los minutos, partidos e identificador de Understat a cada par
pares = pares.merge(
    # Columnas de Understat que nos interesan
    understat[["player", "team", "player_id", "minutes", "matches"]].rename(columns={
        "player": "nombre_understat", "team": "equipo_understat", "player_id": "id_understat",
        "minutes": "minutos_us", "matches": "partidos_us"}),
    # Cruzamos por nombre y equipo de Understat
    on=["nombre_understat", "equipo_understat"], how="left", validate="one_to_one")

# Ningún par puede quedarse sin su fila de Understat
assert pares["id_understat"].notna().all(), "hay pares sin fila de Understat"
# Una fila de Understat no puede estar en dos pares
assert pares["id_understat"].is_unique, "una fila de Understat aparece en dos pares"
# Un jugador de posiciones no puede estar en dos pares
assert not pares.duplicated(["nombre_posiciones", "equipo"]).any(), "un jugador de posiciones aparece dos veces"
# Comprobamos que los nombres de posiciones existen en mi fichero
claves_pos = set(zip(posiciones["jugador"], posiciones["equipo"]))
# Si alguno no existe, paramos
assert all(k in claves_pos for k in zip(pares["nombre_posiciones"], pares["equipo"])), "nombre de posiciones inexistente"
# Mostramos el recuento por origen
print(pares["origen"].value_counts().to_string())

origen
directo          481
recuperado_36     36
traspasado         6


## 5. Jugadores de Understat sin fila en mi fichero de posiciones

Se separan los porteros (fuera de esta versión) de los jugadores de campo, que se recuperan.

In [10]:
# Filas de Understat que no forman parte de ningún par
libres = understat[~understat["player_id"].isin(pares["id_understat"])].copy()
# Un portero es quien tiene GK en la posición de Understat
libres["portero"] = libres["position"].str.contains("GK")

# Jugadores de campo sin posición: entran en el modelo
sin_pos = libres[~libres["portero"]].rename(columns={
    "player": "nombre_understat", "team": "equipo", "player_id": "id_understat",
    "minutes": "minutos_us", "matches": "partidos_us"})
# Su equipo de Understat es también el del modelo
sin_pos["equipo_understat"] = sin_pos["equipo"]
# Marcamos su origen
sin_pos["origen"] = "understat_sin_posicion"

# Mostramos el reparto
print("Understat sin fila en posiciones:", len(libres))
print("  porteros (fuera):", int(libres["portero"].sum()))
print("  jugadores de campo (entran):", len(sin_pos))

Understat sin fila en posiciones: 77
  porteros (fuera): 39
  jugadores de campo (entran): 38


## 6. Jugadores de posiciones que no jugaron

In [11]:
# Nombres de posiciones descartados por cruce dudoso
desc_pos = {(n, e) for n, e, _, _ in DESCARTADOS_POSICIONES}
# Nombres de posiciones que ya tienen par
con_par = set(zip(pares["nombre_posiciones"], pares["equipo"]))
# Los que no jugaron son el resto de mi fichero
sin_minutos = posiciones[[(j, e) not in con_par and (j, e) not in desc_pos
                          for j, e in zip(posiciones["jugador"], posiciones["equipo"])]].copy()

# Control: ninguno debe tener eventos con su nombre en su equipo
# Identificadores que tienen al menos un evento
ids_con_eventos = set(eventos["player_id"].dropna().astype("int64"))
# Buscamos en alineaciones a los que coinciden por nombre y equipo y además tienen eventos
cruce = alineaciones.assign(n=alineaciones["player"].map(normalizar)).merge(
    sin_minutos.assign(n=sin_minutos["jugador"].map(normalizar)), on=["equipo", "n"])
# Nos quedamos con los que tienen eventos
con_eventos = cruce[cruce["player_id"].isin(ids_con_eventos)]["jugador"].unique()
# Mostramos el resultado del control
print("no jugaron:", len(sin_minutos), "· de ellos con eventos en su equipo:", len(con_eventos), list(con_eventos))

no jugaron:

 112 · de ellos con eventos en su equipo: 0 []


## 7. Identificador de WhoScored de cada jugador

Los eventos se cruzan por identificador, nunca por nombre: así los dos David López no se mezclan y
los traspasados suman los eventos de sus dos clubes.

In [12]:
# Los 15 de la tabla del TFM que tenían los eventos vacíos: nombre en WhoScored confirmado
RECUPERADOS_15 = {
    ("Pape Gueye", "Villarreal"): "Pape Gueye",
    ("Pau Cabanes", "Villarreal"): "Pau Cabanes",
    ("Bryan Gil", "Girona"): "Bryan Gil",
    ("Jon Pacheco", "Alaves"): "Jon Pacheco",
    ("Jonny", "Alaves"): "Jonny Otto",
    ("Pablo Barrios", "Atletico Madrid"): "Pablo Barrios",
    ("Miguel Rubio", "Espanyol"): "Miguel Rubio",
    ("Urko González de Zárate", "Espanyol"): "Urko González de Zárate",
    ("Karl Etta Eyong", "Levante"): "Karl Etta Eyong",
    ("Tai Abed", "Levante"): "Tay Abed",
    ("Jorge Cestero", "Real Madrid"): "Cestero",
    ("Kylian Mbappé", "Real Madrid"): "Kylian Mbappé",
    ("Álvaro Leiva", "Real Madrid"): "Ávaro Leiva",
    ("Daniel Díaz", "Real Sociedad"): "Dani Díaz",
    ("Arnaut Danjuma", "Valencia"): "Arnaut Danjuma",
}

# Jugadores de Understat sin posición cuyo nombre en WhoScored es otra grafía: mismo equipo y mismos partidos
RECUPERADOS_GRAFIA = {
    # La tilde y la i/y: 2 partidos en Understat y 2 con eventos
    ("Yeremi Pino", "Villarreal"): "Yéremy Pino",
    # Nombre completo frente a nombre corto: 1 partido en Understat y 1 con eventos
    ("Stanis Idumbo Muzambo", "Sevilla"): "Stanis Idumbo",
}

# Cruces de eventos que no se aplican: el jugador entra con los eventos vacíos
# (nombre en el modelo, equipo, nombre propuesto en WhoScored, motivo)
SIN_EVENTOS = [
    ("Adam El Mokhtari", "Elche", "Adam Boayar",
     "solo comparten el nombre de pila; 7 partidos en Understat y 7 con eventos"),
    ("Rubén Iranzo", "Valencia", "Rubo",
     "único jugador de campo del Valencia con un partido; indicio muy débil"),
    ("Sergio Lozano", "Rayo Vallecano", "Sergio Lozano (Levante)",
     "traspaso dudoso: Understat y los eventos solo lo sitúan en el Levante; puede ser otro jugador"),
    ("Carlos Benavídez", "Alaves", "Carlos Protesoni",
     "fila de Understat de un cruce de posiciones descartado"),
    ("Yago Alonso", "Elche", "Yago Santiago",
     "fila de Understat de un cruce de posiciones descartado"),
]

# Una fila por jugador del modelo: pares con posición y jugadores de campo sin posición
mapa = pd.concat([pares, sin_pos[pares.columns.drop("nombre_posiciones")]], ignore_index=True)
# El nombre que se usa en el modelo: el de mi fichero si existe, si no el de Understat
mapa["nombre_modelo"] = mapa["nombre_posiciones"].fillna(mapa["nombre_understat"])

# Marcamos el origen de los 15 recuperados
es15 = [(n, e) in RECUPERADOS_15 for n, e in zip(mapa["nombre_posiciones"], mapa["equipo"])]
# Cambiamos su origen
mapa.loc[es15, "origen"] = "recuperado_15"
# Mostramos cuántos se marcaron
print("marcados como recuperado_15:", sum(es15))
# Marcamos el origen de los recuperados por grafía
es_grafia = [(n, e) in RECUPERADOS_GRAFIA for n, e in zip(mapa["nombre_modelo"], mapa["equipo"])]
# Cambiamos su origen (siguen siendo jugadores de Understat sin fila en mi fichero de posiciones)
mapa.loc[es_grafia, "origen"] = "recuperado_grafia"
# Mostramos cuántos se marcaron
print("marcados como recuperado_grafia:", sum(es_grafia))

# Catálogo de identificadores: nombre normalizado, equipo e identificador de WhoScored
catalogo = alineaciones[["player_id", "player", "equipo"]].drop_duplicates()
# Normalizamos el nombre del catálogo
catalogo["n"] = catalogo["player"].map(normalizar)

# Jugadores a los que no se les busca identificador
excluir = {(n, e) for n, e, _, _ in SIN_EVENTOS}


# Busca el identificador de un jugador del modelo y devuelve (id, nombre en WhoScored, cómo)
def buscar_id(fila):
    # Clave del jugador en el modelo
    clave = (fila["nombre_modelo"], fila["equipo"])
    # Si está en la lista de exclusión, no se busca
    if clave in excluir:
        return None, None, "excluido"
    # Si es uno de los 15 o de los recuperados por grafía, el nombre de WhoScored está fijado a mano
    if clave in RECUPERADOS_15 or clave in RECUPERADOS_GRAFIA:
        # Nombre fijado en la lista correspondiente
        nombres = {normalizar({**RECUPERADOS_15, **RECUPERADOS_GRAFIA}[clave])}
    else:
        # Probamos con el nombre de posiciones y con el de Understat
        nombres = {normalizar(x) for x in [fila["nombre_posiciones"], fila["nombre_understat"]] if pd.notna(x)}
    # Equipos donde buscar: el del modelo y el de Understat
    equipos = {fila["equipo"], fila["equipo_understat"]}
    # Candidatos del mismo nombre en esos equipos
    c = catalogo[catalogo["n"].isin(nombres) & catalogo["equipo"].isin(equipos)]
    # Si no hay en esos equipos, buscamos en toda la liga (traspasos que Understat no refleja)
    if c.empty:
        c = catalogo[catalogo["n"].isin(nombres)]
        # Anotamos que se encontró fuera de su equipo
        como = "nombre, otro equipo"
    else:
        como = "nombre, mismo equipo"
    # Identificadores distintos encontrados
    ids = c["player_id"].unique()
    # Ninguno: sin identificador
    if len(ids) == 0:
        return None, None, "sin coincidencia"
    # Más de uno: ambiguo, no se elige
    if len(ids) > 1:
        return None, None, "ambiguo"
    # Exactamente uno: lo devolvemos con el nombre que usa WhoScored
    return int(ids[0]), c["player"].iloc[0], como


# Aplicamos la búsqueda a cada jugador del modelo
resultado = mapa.apply(buscar_id, axis=1, result_type="expand")
# Guardamos las tres columnas del resultado
mapa[["id_whoscored", "nombre_whoscored", "como_se_cruzo"]] = resultado
# El identificador es un entero que puede faltar
mapa["id_whoscored"] = mapa["id_whoscored"].astype("Int64")

# Recuento de cómo se cruzó cada jugador, por origen
print(pd.crosstab(mapa["origen"], mapa["como_se_cruzo"], margins=True).to_string())

marcados como recuperado_15: 15
marcados como recuperado_grafia: 2


como_se_cruzo           excluido  nombre, mismo equipo  All
origen                                                     
directo                        2                   464  466
recuperado_15                  0                    15   15
recuperado_36                  0                    36   36
recuperado_grafia              0                     2    2
traspasado                     1                     5    6
understat_sin_posicion         2                    34   36
All                            5                   556  561


In [13]:
# Jugadores encontrados fuera de su equipo: se revisan uno a uno
print(mapa[mapa["como_se_cruzo"] == "nombre, otro equipo"][
    ["nombre_modelo", "equipo", "equipo_understat", "nombre_whoscored", "origen"]].to_string(index=False))

Empty DataFrame
Columns: [nombre_modelo, equipo, equipo_understat, nombre_whoscored, origen]
Index: []


In [14]:
# Jugadores sin identificador que no están en la lista de exclusión
print(mapa[mapa["como_se_cruzo"].isin(["sin coincidencia", "ambiguo"])][
    ["nombre_modelo", "equipo", "nombre_understat", "minutos_us", "origen", "como_se_cruzo"]].to_string(index=False))

Empty DataFrame
Columns: [nombre_modelo, equipo, nombre_understat, minutos_us, origen, como_se_cruzo]
Index: []


In [15]:
# Un identificador de WhoScored no puede pertenecer a dos jugadores del modelo
repetidos = mapa["id_whoscored"].dropna()
# Si hay repetidos, paramos
assert repetidos.is_unique, mapa[mapa["id_whoscored"].isin(repetidos[repetidos.duplicated()])]
# Mostramos el resultado
print("identificadores de WhoScored únicos:", repetidos.nunique())

identificadores de WhoScored únicos: 556


## 8. Acciones de Opta por identificador

- intercepciones, despejes y pases bloqueados: eventos de ese tipo, con cualquier resultado (como en el TFM)
- duelos aéreos: Opta registra un evento `Aerial` por cada jugador del duelo, con resultado `Successful` para
  el que lo gana y `Unsuccessful` para el que lo pierde. **Disputados** = todos; **ganados** = `Successful`.
- entradas: **intentadas** = eventos `Tackle` (llega al balón) más `Challenge` (el rival le regatea, siempre
  `Unsuccessful`); **exitosas** = `Tackle` con resultado `Successful` (su equipo se queda el balón).
- `entradas_tfm`: todos los eventos `Tackle`, que es lo que el TFM llamaba "entradas". Solo para la auditoría.
- centros: pases con el calificador `Cross`
- toques en el área: toques con x ≥ 83 e y entre 21,1 y 78,9 (coordenadas de Opta, 0 a 100)

In [16]:
# Descartamos los eventos sin jugador (inicios de periodo, etc.)
ev = eventos[eventos["player_id"].notna()].copy()
# El identificador pasa a entero
ev["player_id"] = ev["player_id"].astype("int64")

# Tipos de evento de Opta que se cuentan enteros, con cualquier resultado
ACCIONES = {"Interception": "intercepciones", "Clearance": "despejes", "BlockedPass": "pases_bloqueados"}
# Marcamos cada tipo como columna verdadero/falso
for tipo, nombre in ACCIONES.items():
    # Verdadero si el evento es de ese tipo
    ev[nombre] = ev["type"] == tipo

# Resultado del evento según Opta
gana = ev["outcome_type"] == "Successful"
# Duelo aéreo disputado: cualquier evento Aerial
ev["duelos_aereos_disputados"] = ev["type"] == "Aerial"
# Duelo aéreo ganado: evento Aerial con resultado Successful
ev["duelos_aereos_ganados"] = (ev["type"] == "Aerial") & gana
# Entrada intentada: Tackle (llega al balón) o Challenge (el rival le regatea)
ev["entradas_intentadas"] = ev["type"].isin(["Tackle", "Challenge"])
# Entrada exitosa: Tackle con resultado Successful (su equipo recupera la posesión)
ev["entradas_exitosas"] = (ev["type"] == "Tackle") & gana
# Entrada tal y como la contaba el TFM: todos los Tackle (solo para la auditoría)
ev["entradas_tfm"] = ev["type"] == "Tackle"

# Pasamos los calificadores a texto para buscar "Cross"
calificadores = ev["qualifiers"].astype(str)
# Un centro es un pase con el calificador Cross
ev["centros"] = (ev["type"] == "Pass") & calificadores.str.contains("Cross")
# Toque dentro del área rival
ev["toques_area"] = ev["is_touch"].fillna(False).astype(bool) & (ev["x"] >= 83) & ev["y"].between(21.1, 78.9)

# Columnas que se cuentan
CONTEOS = (["entradas_intentadas", "entradas_exitosas"] + list(ACCIONES.values())
           + ["duelos_aereos_disputados", "duelos_aereos_ganados", "centros", "toques_area", "entradas_tfm"])
# Sumamos las acciones de cada jugador en toda la temporada, sea cual sea su club
por_jugador = ev.groupby("player_id")[CONTEOS].sum().astype("int64")
# Partidos distintos en los que el jugador tiene al menos un evento
por_jugador["partidos_eventos"] = ev.groupby("player_id")["game_id"].nunique()

# Partidos con eventos por club, para ver el reparto de los traspasados
por_club = ev.groupby(["player_id", "equipo"])["game_id"].nunique().reset_index()
# Texto del tipo "Celta Vigo 10 + Girona 17", ordenado por nombre de club
por_jugador["equipos_eventos"] = por_club.sort_values("equipo").groupby("player_id").apply(
    lambda g: " + ".join(f"{e} {n}" for e, n in zip(g["equipo"], g["game_id"])))
# Recuperamos el identificador como columna
por_jugador = por_jugador.reset_index().rename(columns={"player_id": "id_whoscored"})

# Control: las cifras deben ser idénticas a las del TFM salvo donde el TFM falló
control = mapa[mapa["origen"] == "directo"].merge(por_jugador, on="id_whoscored")
# Columnas del TFM y sus equivalentes en este proyecto
TFM = {"Tackle": "entradas_tfm", "Interception": "intercepciones", "Clearance": "despejes",
       "BlockedPass": "pases_bloqueados", "Aerial": "duelos_aereos_disputados", "centros": "centros",
       "toques_area": "toques_area"}
# Añadimos las cifras del TFM a cada jugador
control = control.merge(metricas_tfm[["jugador", "equipo"] + list(TFM)].rename(
    columns={"jugador": "nombre_posiciones", **{t: "tfm_" + t for t in TFM}}), on=["nombre_posiciones", "equipo"])
# Jugadores con alguna cifra distinta
distintos = control[[any(f["tfm_" + t] != f[p] for t, p in TFM.items()) for _, f in control.iterrows()]]
# Mostramos el resultado
print("directos comparados con el TFM:", len(control), "· con alguna cifra distinta:", len(distintos))
print(distintos[["nombre_posiciones", "equipo", "tfm_Tackle", "entradas_tfm", "tfm_Interception", "intercepciones",
                 "tfm_Clearance", "despejes"]].to_string(index=False))

directos comparados con el TFM: 464 · con alguna cifra distinta: 2
nombre_posiciones   equipo  tfm_Tackle  entradas_tfm  tfm_Interception  intercepciones  tfm_Clearance  despejes
      David López Mallorca        15.0            11               8.0               8           92.0        81
      David López   Girona        15.0             4               8.0               0           92.0        11


## 9. Posición desde las alineaciones para los jugadores sin posición

Se cuentan sus titularidades por puesto de WhoScored, se traducen con la misma equivalencia que validó
las posiciones del TFM y se agrupan.

**Equivalencia de puestos.** Es la del TFM con un cambio: **FWL y FWR van a extremo**, no a delantero.
El fichero propio de posiciones ya los trataba así: entre sus jugadores, los titulares en FWL son extremos
57 veces y delanteros 10, y en FWR, 46 y 28. La equivalencia del TFM (FWL/FWR = delantero) no la seguía.

**Regla de concluyente:** al menos 3 titularidades y más de la mitad de ellas en el mismo grupo de
posición. Si no, el jugador queda en `sin_confirmar`. Nada se aplica al modelo hasta que David lo
confirme en la lista `CONFIRMADAS`.

In [17]:
# Equivalencia entre los puestos de WhoScored y mis posiciones (la del TFM, con FWL y FWR a extremo)
EQUIVALENCIA = {
    "DC": "central",
    "DL": "lateral", "DR": "lateral", "DML": "lateral", "DMR": "lateral",
    "DMC": "pivote",
    "MC": "mediocentro",
    "AMC": "mediapunta",
    "AML": "extremo", "AMR": "extremo", "ML": "extremo", "MR": "extremo",
    "FW": "delantero",
    # Cambio respecto al TFM: la banda en ataque es extremo, como en mi fichero de posiciones
    "FWL": "extremo", "FWR": "extremo",
}
# Grupos de comparación: pivote y mediocentro van juntos
GRUPOS = {"central": "central", "lateral": "lateral", "pivote": "pivote_mediocentro",
          "mediocentro": "pivote_mediocentro", "mediapunta": "mediapunta", "extremo": "extremo",
          "delantero": "delantero", "sin_definir": "sin_definir"}
# Titularidades mínimas para dar una posición por buena
MIN_TITULARIDADES = 3

# Posiciones confirmadas por David el 16/11/2026: (nombre en Understat, equipo)
CONFIRMADAS = [
    ("Carlos Vicente", "Alaves"), ("Bryan Zaragoza", "Celta Vigo"), ("Salomón Rondón", "Real Oviedo"),
    ("Conor Gallagher", "Atletico Madrid"), ("Yáser Asprilla", "Girona"), ("Josip Brekalo", "Real Oviedo"),
    ("Jorge Cabello", "Levante"), ("Christantus Uche", "Getafe"), ("Oier Luengo", "Real Oviedo"),
    ("Moussa Diarra", "Alaves"),
]

# Titularidades de todos los jugadores (los suplentes no tienen puesto)
titulares = alineaciones[alineaciones["titular"]].copy()
# Traducimos el puesto de WhoScored a mi posición
titulares["posicion"] = titulares["posicion_ws"].map(EQUIVALENCIA)


# Calcula la propuesta de posición de un jugador a partir de sus titularidades
def proponer(fila):
    # Sin identificador no hay alineaciones que mirar
    if pd.isna(fila["id_whoscored"]):
        return pd.Series({"titularidades": 0, "recuento_puestos_ws": "", "posicion_propuesta": None,
                          "grupo_propuesto": None, "concluyente": False, "motivo": "sin identificador de WhoScored"})
    # Titularidades de este jugador
    t = titulares[titulares["player_id"] == fila["id_whoscored"]]
    # Recuento por puesto original de WhoScored, de más a menos
    puestos = t["posicion_ws"].value_counts()
    # Texto del recuento, por ejemplo "DC 12 · DMC 3"
    texto = " · ".join(f"{p} {n}" for p, n in puestos.items())
    # Sin titularidades no se puede deducir nada
    if len(t) == 0:
        return pd.Series({"titularidades": 0, "recuento_puestos_ws": "solo suplente", "posicion_propuesta": None,
                          "grupo_propuesto": None, "concluyente": False, "motivo": "nunca fue titular"})
    # Recuento por grupo de posición
    grupos = t["posicion"].map(GRUPOS).value_counts()
    # Grupo más repetido y su peso
    grupo, n_grupo = grupos.index[0], grupos.iloc[0]
    # Dentro de ese grupo, la posición más repetida
    pos = t[t["posicion"].map(GRUPOS) == grupo]["posicion"].value_counts()
    # Si dentro del grupo hay empate entre posiciones, no elegimos
    posicion = pos.index[0] if (len(pos) == 1 or pos.iloc[0] > pos.iloc[1]) else None
    # Empate entre grupos
    empate = len(grupos) > 1 and grupos.iloc[1] == n_grupo
    # Aplicamos la regla de concluyente
    if len(t) < MIN_TITULARIDADES:
        motivo, ok = f"menos de {MIN_TITULARIDADES} titularidades", False
    elif empate or n_grupo * 2 <= len(t):
        motivo, ok = "ningún grupo supera la mitad de las titularidades", False
    elif posicion is None:
        motivo, ok = "empate entre posiciones del mismo grupo", False
    else:
        motivo, ok = f"{n_grupo} de {len(t)} titularidades en el grupo", True
    # Devolvemos la propuesta
    return pd.Series({"titularidades": len(t), "recuento_puestos_ws": texto,
                      "posicion_propuesta": posicion if ok else None, "grupo_propuesto": grupo if ok else None,
                      "concluyente": ok, "motivo": motivo})


# Jugadores de campo sin posición en mi fichero
pos_ws = mapa[mapa["nombre_posiciones"].isna()].copy()
# Calculamos la propuesta de cada uno
pos_ws = pd.concat([pos_ws, pos_ws.apply(proponer, axis=1)], axis=1)
# Marcamos los confirmados por David
pos_ws["confirmado"] = [(n, e) in CONFIRMADAS for n, e in zip(pos_ws["nombre_understat"], pos_ws["equipo"])]
# Solo se puede confirmar una propuesta concluyente
assert not (pos_ws["confirmado"] & ~pos_ws["concluyente"]).any(), "hay confirmados no concluyentes"
# Todos los confirmados existen
assert pos_ws["confirmado"].sum() == len(CONFIRMADAS), "algún confirmado no se encuentra"
# Ordenamos por minutos, de más a menos
pos_ws = pos_ws.sort_values("minutos_us", ascending=False)
# El identificador de WhoScored es un entero que puede faltar
pos_ws["id_whoscored"] = pos_ws["id_whoscored"].astype("Int64")
# Mostramos la tabla para revisión
print(pos_ws[["nombre_understat", "equipo", "minutos_us", "titularidades", "recuento_puestos_ws",
              "posicion_propuesta", "grupo_propuesto", "concluyente", "confirmado", "motivo"]].to_string(index=False))

     nombre_understat          equipo  minutos_us  titularidades                 recuento_puestos_ws posicion_propuesta    grupo_propuesto  concluyente  confirmado                             motivo
       Carlos Vicente          Alaves        1250             11                MR 8 · AMR 2 · FWR 1            extremo            extremo         True        True 11 de 11 titularidades en el grupo
       Bryan Zaragoza      Celta Vigo        1138             13        FWL 9 · FWR 2 · AMC 1 · ML 1            extremo            extremo         True        True 12 de 13 titularidades en el grupo
       Salomón Rondón     Real Oviedo        1054             13                               FW 13          delantero          delantero         True        True 13 de 13 titularidades en el grupo
      Conor Gallagher Atletico Madrid         630              4                        MC 3 · DMC 1        mediocentro pivote_mediocentro         True        True   4 de 4 titularidades en el grupo
     

## 10. Guardado

In [18]:
# Añadimos al mapa los partidos con eventos de cada jugador
mapa = mapa.merge(por_jugador[["id_whoscored", "partidos_eventos"]], on="id_whoscored", how="left")
# Identificador y partidos son enteros que pueden faltar (no decimales)
mapa[["id_whoscored", "partidos_eventos"]] = mapa[["id_whoscored", "partidos_eventos"]].astype("Int64")
# Columnas del mapa en el orden acordado (más los identificadores de Understat y el equipo de Understat)
COLUMNAS_MAPA = ["nombre_posiciones", "nombre_understat", "nombre_whoscored", "id_whoscored", "equipo",
                 "minutos_us", "partidos_us", "partidos_eventos", "origen",
                 "id_understat", "equipo_understat"]
# Ordenamos por equipo y nombre
mapa = mapa.sort_values(["equipo", "nombre_modelo"])
# Guardamos el mapa de nombres
mapa[COLUMNAS_MAPA].to_csv("data/mapa_nombres.csv", index=False)

# Guardamos solo las acciones de los jugadores del modelo
por_jugador[por_jugador["id_whoscored"].isin(mapa["id_whoscored"])].to_csv(
    "data/eventos_por_jugador.csv", index=False)

# Guardamos las posiciones deducidas de las alineaciones
pos_ws[["nombre_understat", "equipo", "id_understat", "id_whoscored", "minutos_us", "titularidades",
        "recuento_puestos_ws", "posicion_propuesta", "grupo_propuesto", "concluyente", "motivo",
        "confirmado"]].to_csv("data/posiciones_alineaciones.csv", index=False)

# Guardamos los que no jugaron
sin_minutos[["jugador", "equipo", "posicion"]].sort_values(["equipo", "jugador"]).to_csv(
    "data/sin_minutos.csv", index=False)

# Cruces descartados de posiciones: el jugador de posiciones no entra
d1 = pd.DataFrame(DESCARTADOS_POSICIONES, columns=["nombre", "equipo", "nombre_propuesto", "motivo"])
# Tipo de cruce y efecto en el modelo
d1["cruce"], d1["efecto"] = "posiciones-understat", "fila de posiciones fuera; la fila de Understat entra sin posición y sin eventos"
# Cruces descartados de eventos: el jugador entra con eventos vacíos
d2 = pd.DataFrame(SIN_EVENTOS, columns=["nombre", "equipo", "nombre_propuesto", "motivo"])
# Tipo de cruce y efecto en el modelo
d2["cruce"], d2["efecto"] = "eventos", "entra en el modelo con las métricas de eventos vacías"
# Juntamos las dos listas
descartados = pd.concat([d1, d2], ignore_index=True)
# Minutos de Understat por nombre y equipo de Understat
min_us = dict(zip(zip(understat["player"], understat["team"]), understat["minutes"]))
# Minutos de Understat por nombre y equipo del modelo
min_modelo = dict(zip(zip(mapa["nombre_modelo"], mapa["equipo"]), mapa["minutos_us"]))
# Los descartados de posiciones toman los minutos de la fila de Understat propuesta; el resto, los suyos
descartados["minutos_us"] = [min_us[(p, e)] if c == "posiciones-understat" else min_modelo[(n, e)]
                             for n, e, p, c in zip(descartados["nombre"], descartados["equipo"],
                                                   descartados["nombre_propuesto"], descartados["cruce"])]
# Guardamos la lista
descartados[["nombre", "equipo", "cruce", "nombre_propuesto", "minutos_us", "motivo", "efecto"]].to_csv(
    "data/cruces_descartados.csv", index=False)

# Resumen de lo guardado
print("mapa_nombres.csv:", len(mapa), "jugadores")
print("eventos_por_jugador.csv:", int(por_jugador["id_whoscored"].isin(mapa["id_whoscored"]).sum()), "jugadores")
print("posiciones_alineaciones.csv:", len(pos_ws), "jugadores")
print("sin_minutos.csv:", len(sin_minutos), "jugadores")
print("cruces_descartados.csv:", len(descartados), "cruces")

mapa_nombres.csv:

 561 jugadores
eventos_por_jugador.csv: 556 jugadores
posiciones_alineaciones.csv: 38 jugadores
sin_minutos.csv: 112 jugadores
cruces_descartados.csv: 7 cruces
